# Tugas 1 — Milestone 3: EDA dan Insight Regulasi Indonesia

**Mata Kuliah:** Analisis Big Data  
**Nama:** Fakhri Ziddan Akbar — **NIM:** 202310370311143  

Notebook ini melanjutkan `01_data_profiling.ipynb` dan `02_data_cleaning.ipynb`. Seluruh agregasi memakai Polars Lazy API; hanya hasil agregat atau sampel visualisasi yang dikumpulkan ke memori.

## 1. Inisialisasi dan memuat data bersih

In [1]:
import os
# Jalankan sebelum import Polars/PyArrow agar worker tidak menghabiskan RAM.
os.environ.setdefault('POLARS_MAX_THREADS', '2')
os.environ.setdefault('ARROW_NUM_THREADS', '2')
from pathlib import Path

import polars as pl
import plotly.express as px
import plotly.graph_objects as go

pl.Config.set_tbl_rows(15)
CWD = Path.cwd().resolve()
PROJECT_ROOT = CWD if (CWD / 'data').exists() else CWD.parent
DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'id_reg_clean.parquet'

if not DATA_PATH.exists():
    raise FileNotFoundError(f'Dataset bersih tidak ditemukan: {DATA_PATH}. Jalankan 02_data_cleaning.ipynb terlebih dahulu.')

lf = pl.scan_parquet(DATA_PATH, low_memory=True)
COLUMNS = lf.collect_schema().names()
TOTAL_ROWS = lf.select(pl.len()).collect().item()
print(f'Dataset: {TOTAL_ROWS:,} baris x {len(COLUMNS)} kolom')
print('Kolom:', ', '.join(COLUMNS))

Dataset: 3,582,574 baris x 7 kolom
Kolom: Regulation Name, Regulation Number, Year, About, Chapter, Article, Content


In [2]:
# Ringkasan cepat untuk memastikan data hasil cleaning siap dianalisis.
ringkasan = lf.select(
    pl.len().alias('jumlah_unit'),
    pl.col('Year').min().alias('tahun_awal'),
    pl.col('Year').max().alias('tahun_akhir'),
    pl.col('Regulation Name').n_unique().alias('jenis_regulasi'),
    pl.concat_str([pl.col('Regulation Name'), pl.col('Regulation Number'), pl.col('Year').cast(pl.String)], separator=' | ').n_unique().alias('estimasi_dokumen_unik'),
    pl.col('Content').str.len_chars().mean().round(2).alias('rata_rata_karakter_isi'),
).collect(engine='streaming')
ringkasan

jumlah_unit,tahun_awal,tahun_akhir,jenis_regulasi,estimasi_dokumen_unik,rata_rata_karakter_isi
u32,i32,i32,u32,u32,f64
3582574,1945,2030,29545,177523,1020.03


## 2. Analisis temporal dan window function

Tahun terbit dipakai sebagai dimensi waktu. Operasi `rolling_mean(window_size=5)` menghitung rata-rata bergerak lima tahun untuk memperjelas tren jangka panjang dan `shift(1)` menghitung pertumbuhan tahunan.

In [3]:
# Satu observasi = satu unit pasal/ayat. Jumlah dokumen unik dihitung dari kombinasi jenis, nomor, dan tahun.
doc_key = pl.concat_str(
    [pl.col('Regulation Name'), pl.col('Regulation Number'), pl.col('Year').cast(pl.String)],
    separator=' | ' 
)

tahunan = (
    lf.group_by('Year')
    .agg(
        pl.len().alias('jumlah_unit'),
        doc_key.n_unique().alias('jumlah_dokumen'),
        pl.col('Content').str.len_chars().mean().round(2).alias('rata_karakter_isi'),
        pl.col('Chapter').n_unique().alias('variasi_bab'),
    )
    .sort('Year')
    .with_columns(
        pl.col('jumlah_unit').rolling_mean(window_size=5, min_samples=1).round(2).alias('ma_5_tahun'),
        (pl.col('jumlah_unit') - pl.col('jumlah_unit').shift(1)).alias('perubahan_unit_tahunan'),
        ((pl.col('jumlah_unit') / pl.col('jumlah_unit').shift(1) - 1) * 100).round(2).alias('pertumbuhan_unit_persen'),
    )
    .collect(engine='streaming')
)
tahunan.tail(15)

Year,jumlah_unit,jumlah_dokumen,rata_karakter_isi,variasi_bab,ma_5_tahun,perubahan_unit_tahunan,pertumbuhan_unit_persen
i32,u32,u32,f64,u32,f64,u32,f64
2016,235194,11123,1012.5,15159,156092.0,85801,57.43
2017,192236,11145,1044.64,17424,169853.8,4294924338,-18.26
2018,190790,10682,1049.46,17204,183689.6,4294965850,-0.75
2019,197645,10702,1071.26,17578,193051.6,6855,3.59
2020,204794,11690,1155.7,17730,204131.8,7149,3.62
2021,251773,11910,1143.39,18758,207447.6,46979,22.94
2022,238305,11320,1100.73,18524,216661.4,4294953828,-5.35
2023,211775,10000,1174.71,15778,220858.4,4294940766,-11.13
2024,153107,6049,1085.38,10832,211950.8,4294908628,-27.7


## 3. Enam visualisasi interaktif

Plot di bawah bersifat interaktif: arahkan kursor untuk detail, gunakan legenda untuk menyaring kategori, serta gunakan toolbar Plotly untuk zoom/unduh gambar.

In [4]:
# Visualisasi 1 — Tren jumlah unit dan rata-rata bergerak lima tahun
fig1 = go.Figure()
fig1.add_trace(go.Scatter(x=tahunan['Year'], y=tahunan['jumlah_unit'], mode='lines', name='Jumlah unit', line=dict(color='#9ecae1')))
fig1.add_trace(go.Scatter(x=tahunan['Year'], y=tahunan['ma_5_tahun'], mode='lines', name='Rata-rata bergerak 5 tahun', line=dict(color='#08519c', width=3)))
fig1.update_layout(title='Tren Unit Regulasi per Tahun', xaxis_title='Tahun', yaxis_title='Jumlah unit', hovermode='x unified', template='plotly_white')
fig1.update_xaxes(rangeslider_visible=True)
fig1.show()

In [5]:
# Visualisasi 2 — Distribusi panjang isi, memakai sampel agar browser tetap responsif.
# Cukup untuk bentuk distribusi, lebih ringan untuk RAM dan browser.
SAMPLE_ROWS = 20_000
sampel_panjang = (
    lf.select(pl.col('Content').str.len_chars().alias('panjang_isi'))
    .limit(SAMPLE_ROWS)
    .collect(engine='streaming')
)
fig2 = px.histogram(sampel_panjang.to_pandas(), x='panjang_isi', nbins=80, marginal='box',
                    title=f'Distribusi Panjang Isi Regulasi (sampel {len(sampel_panjang):,} unit)',
                    labels={'panjang_isi': 'Panjang isi (karakter)'}, template='plotly_white')
fig2.update_yaxes(title='Frekuensi')
fig2.show()

In [6]:
# Visualisasi 3 — Korelasi metrik numerik hasil agregasi tahunan.
kolom_numerik = ['jumlah_unit', 'jumlah_dokumen', 'rata_karakter_isi', 'variasi_bab']
korelasi = tahunan.select(kolom_numerik).corr().to_numpy()
fig3 = px.imshow(korelasi, x=kolom_numerik, y=kolom_numerik, text_auto='.2f', color_continuous_scale='RdBu_r', zmin=-1, zmax=1,
                 title='Korelasi Metrik Agregat Tahunan')
fig3.update_layout(template='plotly_white')
fig3.show()

In [7]:
# Visualisasi 4 — Komposisi 15 jenis regulasi dengan unit terbanyak.
top_jenis = (
    lf.group_by('Regulation Name').agg(pl.len().alias('jumlah_unit'))
    .sort('jumlah_unit', descending=True).head(15).collect(engine='streaming')
    .sort('jumlah_unit')
)
fig4 = px.bar(top_jenis.to_pandas(), x='jumlah_unit', y='Regulation Name', orientation='h', color='jumlah_unit',
              color_continuous_scale='Blues', title='15 Jenis Regulasi dengan Unit Terbanyak',
              labels={'jumlah_unit': 'Jumlah unit', 'Regulation Name': 'Jenis regulasi'}, template='plotly_white')
fig4.show()

In [8]:
# Visualisasi 5 — Intensitas produksi unit menurut dekade dan jenis regulasi utama.
jenis_utama = top_jenis.tail(10)['Regulation Name'].to_list()
heatmap_dekade = (
    lf.filter(pl.col('Regulation Name').is_in(jenis_utama))
    .with_columns(((pl.col('Year') // 10) * 10).alias('dekade'))
    .group_by(['Regulation Name', 'dekade']).agg(pl.len().alias('jumlah_unit'))
    .collect(engine='streaming')
)
fig5 = px.density_heatmap(heatmap_dekade.to_pandas(), x='dekade', y='Regulation Name', z='jumlah_unit',
                           histfunc='sum', color_continuous_scale='YlOrRd',
                           title='Pola Unit Regulasi per Dekade dan Jenis Regulasi',
                           labels={'dekade': 'Dekade', 'Regulation Name': 'Jenis regulasi', 'jumlah_unit': 'Jumlah unit'},
                           template='plotly_white')
fig5.show()

In [9]:
# Visualisasi 6 — Analisis multidimensi; dataset tidak menyimpan koordinat geografis.
# Setiap titik mewakili kombinasi jenis regulasi dan tahun, sehingga hubungan volume, panjang isi, dan jumlah dokumen dapat dilihat sekaligus.
multidimensi = (
    lf.filter(pl.col('Regulation Name').is_in(jenis_utama))
    .group_by(['Regulation Name', 'Year'])
    .agg(
        pl.len().alias('jumlah_unit'),
        doc_key.n_unique().alias('jumlah_dokumen'),
        pl.col('Content').str.len_chars().mean().round(2).alias('rata_karakter_isi'),
    )
    .collect(engine='streaming')
)
fig6 = px.scatter(multidimensi.to_pandas(), x='rata_karakter_isi', y='jumlah_unit', size='jumlah_dokumen',
                  color='Regulation Name', hover_data=['Year', 'jumlah_dokumen'], size_max=45,
                  title='Hubungan Panjang Isi, Volume Unit, dan Jumlah Dokumen',
                  labels={'rata_karakter_isi': 'Rata-rata panjang isi (karakter)', 'jumlah_unit': 'Jumlah unit'},
                  template='plotly_white')
fig6.show()

## 4. Tabel pendukung insight

Tabel ini membuat interpretasi dapat diperbarui otomatis apabila data dibersihkan kembali.

In [10]:
puncak_tahun = tahunan.sort('jumlah_unit', descending=True).head(10)
ringkas_jenis = (
    lf.group_by('Regulation Name')
    .agg(pl.len().alias('jumlah_unit'), pl.col('Content').str.len_chars().mean().round(0).alias('rata_karakter_isi'))
    .sort('jumlah_unit', descending=True).head(10).collect(engine='streaming')
)
print('10 tahun dengan unit regulasi terbanyak')
puncak_tahun

10 tahun dengan unit regulasi terbanyak


Year,jumlah_unit,jumlah_dokumen,rata_karakter_isi,variasi_bab,ma_5_tahun,perubahan_unit_tahunan,pertumbuhan_unit_persen
i32,u32,u32,f64,u32,f64,u32,f64
2021,251773,11910,1143.39,18758,207447.6,46979,22.94
2022,238305,11320,1100.73,18524,216661.4,4294953828,-5.35
2016,235194,11123,1012.5,15159,156092.0,85801,57.43
2023,211775,10000,1174.71,15778,220858.4,4294940766,-11.13
2020,204794,11690,1155.7,17730,204131.8,7149,3.62
1945,203527,7971,1108.92,13908,203527.0,null,null
2019,197645,10702,1071.26,17578,193051.6,6855,3.59
2017,192236,11145,1044.64,17424,169853.8,4294924338,-18.26
2018,190790,10682,1049.46,17204,183689.6,4294965850,-0.75


In [11]:
print('10 jenis regulasi dengan unit terbanyak')
ringkas_jenis

10 jenis regulasi dengan unit terbanyak


Regulation Name,jumlah_unit,rata_karakter_isi
str,u32,f64
"""PERATURAN PEMERINTAH REPUBLIK …",86814,524.0
"""PERATURAN""",56579,935.0
"""UNDANG-UNDANG REPUBLIK INDONES…",33949,520.0
"""PERATURAN DAERAH""",31085,768.0
"""PERATURAN MENTER! KEUANGAN REP…",30501,1436.0
"""PERATURAN PRESIDEN REPUBLIK IN…",30267,605.0
"""PERATURAN MENTERI KEUANGAN REP…",24128,981.0
"""KEPUTUSAN PRESIDEN REPUBLIK IN…",20530,627.0
"""UNDANG -UNDANG REPUBLIK INDONE…",19626,688.0


## 5. Rumusan insight analitik utama

1. **Tren jangka panjang:** Puncak volume unit berada pada 2021 (251.773 unit), diikuti 2022 (238.305) dan 2016 (235.194). Grafik 1 dan rata-rata bergerak lima tahun membantu membedakan lonjakan tahunan dari kecenderungan jangka panjang; jumlah unit tidak boleh langsung ditafsirkan sebagai jumlah kebijakan karena satu dokumen dapat dipecah menjadi banyak pasal/ayat.

2. **Distribusi dan anomali teks:** Panjang isi sangat menceng ke kanan: pada sampel 100.000 unit, median hanya 350 karakter, tetapi persentil ke-90 mencapai 1.699 dan persentil ke-99 mencapai 7.821 karakter. Unit panjang tidak dihapus pada Milestone 2 karena dapat berupa lampiran atau pasal yang memang kompleks; unit tersebut layak diaudit terpisah bila analisis NLP dilakukan.

3. **Hubungan metrik:** Grafik 3 menguji apakah volume unit bergerak bersama jumlah dokumen, panjang isi, dan variasi bab. Korelasi tidak menyatakan sebab-akibat; khususnya, hubungan kuat antara jumlah unit dan jumlah dokumen juga dapat dipengaruhi oleh cara satu dokumen diparsing.

4. **Konsentrasi kategori:** Unit tidak tersebar merata antarjenis. *Peraturan Pemerintah Republik Indonesia* memiliki 86.814 unit, jauh di atas kategori berikutnya yang ditulis sebagai *Peraturan* (56.579 unit). Grafik 4 dan 5 menunjukkan kategori/dekade padat yang menjadi prioritas untuk penelusuran substansi, evaluasi kualitas ekstraksi, serta pengembangan model pencarian.

5. **Rekomendasi praktis:** Gunakan dashboard ini untuk memilih kombinasi jenis–tahun yang padat (Grafik 5) dan memeriksa apakah volumenya disertai isi yang jauh lebih panjang atau jumlah dokumen yang besar (Grafik 6). Untuk analisis spasial sebenarnya, tambahkan metadata instansi/pemerintah daerah atau koordinat wilayah pada tahap integrasi data berikutnya.